In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.tools
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn import metrics

df = pd.read_csv('Life Expectancy Data.csv')

# 1. EDA

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df['Measles'].unique()

### Data quality checks

In [ ]:
checks = {
    'missing values': df.isna().sum().sum(),
    'duplicate rows': df.duplicated().sum(),
    'duplicate country-year': df.duplicated(subset=['Country', 'Year']).sum(),
    'countries': df.Country.nunique(),
    'years per country': df.groupby('Country').size().unique().tolist(),
}
for k, v in checks.items():
    print(f'{k:24s} {v}')

In [ ]:
impossible = {
    'life expectancy outside 20-100': ((df.Life_expectancy < 20) | (df.Life_expectancy > 100)).sum(),
    'immunisation outside 0-100': ((df[['Hepatitis_B', 'Polio', 'Diphtheria', 'Measles']] > 100)
                                   | (df[['Hepatitis_B', 'Polio', 'Diphtheria', 'Measles']] < 0)).any(axis=1).sum(),
    'thinness above 100': (df[['Thinness_ten_nineteen_years', 'Thinness_five_nine_years']] > 100).any(axis=1).sum(),
    'infant deaths > under five': (df.Infant_deaths > df.Under_five_deaths).sum(),
    'negative values': int((df.select_dtypes('number') < 0).any().any()),
    'GDP <= 0': (df.GDP_per_capita <= 0).sum(),
}
for k, v in impossible.items():
    print(f'{k:34s} {v}')

### Outliers

In [ ]:
num_cols = df.select_dtypes('number').columns.drop(
    ['Year', 'Economy_status_Developed', 'Economy_status_Developing'])

Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

outliers = ((df[num_cols] < Q1 - 1.5 * IQR) | (df[num_cols] > Q3 + 1.5 * IQR)).sum()
outliers[outliers > 0].sort_values(ascending=False)

In [ ]:
iqr_mask = ((df[num_cols] < Q1 - 1.5 * IQR) | (df[num_cols] > Q3 + 1.5 * IQR)).any(axis=1)

print(f'rows an IQR rule would delete: {iqr_mask.sum()} ({iqr_mask.mean() * 100:.1f}%)')
print()
print(df[iqr_mask].Region.value_counts().to_string())
print()
print('mean life expectancy  kept:', round(df[~iqr_mask].Life_expectancy.mean(), 1),
      '| deleted:', round(df[iqr_mask].Life_expectancy.mean(), 1))

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(20, 10))

for ax, col in zip(axes.flat, num_cols):
    sns.boxplot(y=df[col], ax=ax, color='steelblue')
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('')

for ax in axes.flat[len(num_cols):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

### Correlations

In [ ]:
c = df.corr(numeric_only=True)
mask = np.triu(np.ones(c.shape), k=1).astype(bool)
pairs = c.where(mask).stack()

pairs[pairs.abs() > 0.75].sort_values(key=abs, ascending=False)

In [ ]:
corrs = df.corr(numeric_only=True)['Life_expectancy'].drop('Life_expectancy')
corrs.sort_values(key=abs, ascending=False).round(3)

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='Region', y='Life_expectancy')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
obj_cols = df.select_dtypes('object').columns
for c in obj_cols:
    print(f'{c:18s} {df[c].nunique():3d}  {sorted(df[c].unique())[:8]}')

# 2. Encoding

In [ ]:
df_clean = df.copy()
df_clean = pd.get_dummies(df_clean, columns=['Region'], drop_first=True)
df_clean.head()

# 3. Feature Engineering

In [ ]:
df_clean['Vaccination_coverage'] = df_clean[['Hepatitis_B', 'Polio', 'Diphtheria', 'Measles']].mean(axis=1)

df_clean = df_clean.drop(columns=['Hepatitis_B', 'Polio', 'Diphtheria', 'Measles'])

In [ ]:
df_clean = df_clean.drop(columns=['Infant_deaths'])

In [ ]:
df_clean = df_clean.drop(columns=['Economy_status_Developing'])

In [ ]:
df_clean['Thinness_avg'] = df_clean[['Thinness_ten_nineteen_years',
                                     'Thinness_five_nine_years']].mean(axis=1)
df_clean = df_clean.drop(columns=['Thinness_ten_nineteen_years',
                                  'Thinness_five_nine_years'])

In [ ]:
df_clean['log_GDP_per_capita'] = np.log1p(df_clean['GDP_per_capita'])
df_clean = df_clean.drop(columns=['GDP_per_capita'])

In [ ]:
df_clean['log_Incidents_HIV'] = np.log1p(df_clean['Incidents_HIV'])
df_clean = df_clean.drop(columns=['Incidents_HIV'])

### Which columns to log

In [ ]:
for col in ['GDP_per_capita', 'Incidents_HIV', 'Population_mln', 'BMI',
            'Under_five_deaths', 'Adult_mortality']:
    logged = np.log1p(df[col])
    print(f'{col:22s} skew {df[col].skew():6.2f} -> {logged.skew():6.2f}   '
          f'r {df[col].corr(df.Life_expectancy):+.3f} -> {logged.corr(df.Life_expectancy):+.3f}')

In [ ]:
col = 'GDP_per_capita'

plt.scatter(df[col], df['Life_expectancy'], s=5, alpha=0.3)
plt.show()

plt.scatter(np.log1p(df[col]), df['Life_expectancy'], s=5, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 14))

for ax, col in zip(axes.flat, num_cols):
    ax.scatter(df[col], df['Life_expectancy'], s=4, alpha=0.3)
    ax.set_title(f'{col}  r={df[col].corr(df.Life_expectancy):.2f}', fontsize=9)

for ax in axes.flat[len(num_cols):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
print(f'Before: {df.shape} -> After: {df_clean.shape}')

In [ ]:
df_clean.head()

# 4. Train/Test Split

In [ ]:
y = df_clean['Life_expectancy']
X = df_clean.drop(columns=['Life_expectancy', 'Country'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}   Test: {len(X_test)}   Features: {X.shape[1]}')

# 5. Scaling

In [ ]:
X_cols = X.columns.tolist()
X_train[X_cols].describe().loc[['min', 'max', 'mean', 'std']].T.round(1)

In [ ]:
scalers = {'unscaled': None,
           'StandardScaler': StandardScaler(),
           'MinMaxScaler': MinMaxScaler(),
           'RobustScaler': RobustScaler()}

rows = []
for name, sc in scalers.items():
    Xtr, Xte = X_train.copy().astype(float), X_test.copy().astype(float)

    if sc is not None:
        Xtr[X_cols] = sc.fit_transform(Xtr[X_cols])
        Xte[X_cols] = sc.transform(Xte[X_cols])

    lr = LinearRegression().fit(Xtr, y_train)
    pred = lr.predict(Xte)

    cond = sm.OLS(y_train, sm.add_constant(Xtr, has_constant='add')).fit().condition_number

    rows.append([name,
                 metrics.r2_score(y_test, pred),
                 metrics.root_mean_squared_error(y_test, pred),
                 metrics.mean_absolute_error(y_test, pred),
                 cond])

scaler_comparison = pd.DataFrame(rows, columns=['scaler', 'r2', 'rmse', 'mae', 'condition_no'])
scaler_comparison.round(6)

In [ ]:
scaler = StandardScaler()

X_train_s = X_train.copy().astype(float)
X_test_s  = X_test.copy().astype(float)
X_train_s[X_cols] = scaler.fit_transform(X_train_s[X_cols])
X_test_s[X_cols]  = scaler.transform(X_test_s[X_cols])

# 6. Fitting the Model

In [ ]:
def fit_ols(Xtr, ytr, Xte):
    """Add the constant, fit, and return the results object plus train/test predictions."""
    Xtr = sm.add_constant(Xtr.astype(float), has_constant='add')
    Xte = sm.add_constant(Xte.astype(float), has_constant='add')[Xtr.columns]
    res = sm.OLS(ytr, Xtr).fit()
    return res, res.predict(Xtr), res.predict(Xte)


res_full, ptr_full, pte_full = fit_ols(X_train_s, y_train, X_test_s)
print('Train R-squared      :', round(res_full.rsquared, 4))
print('Adjusted R-squared   :', round(res_full.rsquared_adj, 4))
print('Condition number     : %.4g' % res_full.condition_number)

In [ ]:
res_full.summary()

### Minimal model

In [ ]:
REGION_COLS = [c for c in X_cols if c.startswith('Region_')]

MINIMAL = ['log_GDP_per_capita', 'Population_mln', 'Schooling',
           'Economy_status_Developed', 'Year'] + REGION_COLS

res_min, ptr_min, pte_min = fit_ols(X_train_s[MINIMAL], y_train, X_test_s[MINIMAL])
print('RMSE:', round(metrics.root_mean_squared_error(y_test, pte_min), 3))
print('MAE :', round(metrics.mean_absolute_error(y_test, pte_min), 3))
print('R2  :', round(metrics.r2_score(y_test, pte_min), 3))

### Collinearity

In [ ]:
design = sm.add_constant(X_train_s.astype(float), has_constant='add')
print('columns:', design.shape[1], ' rank:', np.linalg.matrix_rank(design.values))

In [ ]:
Xv = sm.add_constant(X_train_s[X_cols].astype(float), has_constant='add')

vif = pd.Series([variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1])],
                index=Xv.columns)

vif.drop('const').sort_values(ascending=False).round(2)

In [ ]:
c = X_train.corr()
pairs = c.where(np.triu(np.ones(c.shape), k=1).astype(bool)).stack()
pairs[pairs.abs() > 0.7].sort_values(key=abs, ascending=False)

### Coefficient signs vs simple correlations

In [ ]:
rows = []
for col in X_cols:
    marginal = df_clean[col].corr(df_clean['Life_expectancy'])
    coef = res_full.params[col]
    flipped = (marginal > 0.05 and coef < 0) or (marginal < -0.05 and coef > 0)
    rows.append([col, round(marginal, 3), round(coef, 3),
                 round(res_full.pvalues[col], 4), 'FLIP' if flipped else ''])

sign_check = pd.DataFrame(rows, columns=['feature', 'marginal_r', 'coef', 'p', 'sign'])
sign_check.sort_values('sign', ascending=False)

# 7. Metrics

In [ ]:
def all_metrics(y_true, y_pred, label):
    return {
        'model': label,
        'rmse': metrics.root_mean_squared_error(y_true, y_pred),
        'mae':  metrics.mean_absolute_error(y_true, y_pred),
        'r2':   metrics.r2_score(y_true, y_pred),
    }


model_comparison = pd.DataFrame([
    all_metrics(y_test, pte_full, 'Advanced (all features)'),
    all_metrics(y_test, pte_min, 'Minimal (no health data)'),
])
model_comparison.round(3)

### Log-transformed target

In [ ]:
res_log, ptr_log, pte_log = fit_ols(X_train_s, np.log(y_train), X_test_s)

pred_life = np.exp(pte_log)

comparison = pd.DataFrame([
    all_metrics(y_test, pte_full, 'OLS on LifeExp'),
    all_metrics(y_test, pred_life, 'OLS on log(LifeExp)'),
])
comparison.round(3)

In [ ]:
res_log_min, _, pte_log_min = fit_ols(X_train_s[MINIMAL], np.log(y_train), X_test_s[MINIMAL])

comparison_min = pd.DataFrame([
    all_metrics(y_test, pte_min, 'Minimal on LifeExp'),
    all_metrics(y_test, np.exp(pte_log_min), 'Minimal on log(LifeExp)'),
])
comparison_min.round(3)

### Leave-one-out feature importance

In [ ]:
base_rmse = metrics.root_mean_squared_error(y_test, pte_full)

rows = []
for col in X_cols:
    _, _, pte = fit_ols(X_train_s.drop(columns=[col]), y_train, X_test_s.drop(columns=[col]))
    rmse = metrics.root_mean_squared_error(y_test, pte)
    rows.append([col, rmse, rmse - base_rmse])

drop_test = pd.DataFrame(rows, columns=['dropped', 'rmse', 'change'])
drop_test.sort_values('change', ascending=False).round(4)

### Error by region and by life expectancy band

In [ ]:
errors = pd.DataFrame({
    'Country': df.loc[X_test.index, 'Country'].values,
    'Region': df.loc[X_test.index, 'Region'].values,
    'Year': df.loc[X_test.index, 'Year'].values,
    'actual': y_test.values,
    'advanced': pte_full.values,
    'minimal': pte_min.values,
})
errors['adv_err'] = errors.advanced - errors.actual
errors['min_err'] = errors.minimal - errors.actual

by_region = errors.groupby('Region').apply(lambda d: pd.Series({
    'n': len(d),
    'advanced': np.sqrt((d.adv_err ** 2).mean()),
    'minimal': np.sqrt((d.min_err ** 2).mean()),
}), include_groups=False)
by_region['times_worse'] = by_region.minimal / by_region.advanced
by_region.sort_values('minimal', ascending=False).round(2)

In [ ]:
band = pd.qcut(errors.actual, 4, labels=['lowest 25%', 'second', 'third', 'highest 25%'])

by_band = errors.groupby(band, observed=True).apply(lambda d: pd.Series({
    'advanced': np.sqrt((d.adv_err ** 2).mean()),
    'minimal': np.sqrt((d.min_err ** 2).mean()),
}), include_groups=False)
by_band['times_worse'] = by_band.minimal / by_band.advanced
by_band.round(2)

In [ ]:
errors.reindex(errors.min_err.abs().sort_values(ascending=False).index).head(10)[
    ['Country', 'Year', 'actual', 'advanced', 'minimal']].round(1)

# 8. Predictions and Minimal Model

In [ ]:
VALID_REGIONS = sorted(df['Region'].unique())
VALID_REGIONS

In [ ]:
def predict_life_expectancy(region, year, adult_mortality, under_five_deaths,
                            hepatitis_b, polio, diphtheria, measles,
                            bmi, incidents_hiv, alcohol_consumption,
                            thinness_10_19, thinness_5_9,
                            gdp_per_capita, population_mln,
                            schooling, economy_status_developed):

    if region not in VALID_REGIONS:
        raise ValueError(f"Unknown region '{region}'. Expected one of {VALID_REGIONS}")

    row = {
        'Year': year,
        'Adult_mortality': adult_mortality,
        'Under_five_deaths': under_five_deaths,
        'Alcohol_consumption': alcohol_consumption,
        'BMI': bmi,
        'Population_mln': population_mln,
        'Schooling': schooling,
        'Economy_status_Developed': economy_status_developed,
        'Vaccination_coverage': np.mean([hepatitis_b, polio, diphtheria, measles]),
        'Thinness_avg': np.mean([thinness_10_19, thinness_5_9]),
        'log_GDP_per_capita': np.log1p(gdp_per_capita),
        'log_Incidents_HIV': np.log1p(incidents_hiv),
    }

    row[f'Region_{region}'] = 1

    X_new = pd.DataFrame([row]).reindex(columns=X_cols, fill_value=0).astype(float)
    X_new[X_cols] = scaler.transform(X_new[X_cols])
    X_new = sm.add_constant(X_new, has_constant='add')

    return float(res_full.predict(X_new).iloc[0])


predict_life_expectancy(
    region='Asia',
    year=2015,
    adult_mortality=223,
    under_five_deaths=13,
    hepatitis_b=97,
    polio=97,
    diphtheria=97,
    measles=65,
    bmi=26,
    incidents_hiv=0.1,
    alcohol_consumption=1.32,
    thinness_10_19=4.9,
    thinness_5_9=4.8,
    gdp_per_capita=10000,
    population_mln=78.53,
    schooling=5,
    economy_status_developed=0)

In [ ]:
def predict_life_expectancy_min(region, year, gdp_per_capita, population_mln,
                                schooling, economy_status_developed):

    if region not in VALID_REGIONS:
        raise ValueError(f"Unknown region '{region}'. Expected one of {VALID_REGIONS}")

    row = {
        'Year': year,
        'Population_mln': population_mln,
        'Schooling': schooling,
        'Economy_status_Developed': economy_status_developed,
        'log_GDP_per_capita': np.log1p(gdp_per_capita),
    }

    row[f'Region_{region}'] = 1

    X_new = pd.DataFrame([row]).reindex(columns=X_cols, fill_value=0).astype(float)
    X_new[X_cols] = scaler.transform(X_new[X_cols])
    X_new = sm.add_constant(X_new[MINIMAL], has_constant='add')

    return float(res_min.predict(X_new).iloc[0])


predict_life_expectancy_min(
    region='Asia',
    year=2015,
    gdp_per_capita=10000,
    population_mln=78.53,
    schooling=5,
    economy_status_developed=0)

### Checking both functions against a real test row

In [ ]:
r = df.loc[X_test.index[0]]

print(f'{r.Country} {r.Year}   actual {r.Life_expectancy:.1f}')
print()
print('advanced:', round(predict_life_expectancy(
    region=r.Region, year=r.Year, adult_mortality=r.Adult_mortality,
    under_five_deaths=r.Under_five_deaths, hepatitis_b=r.Hepatitis_B,
    polio=r.Polio, diphtheria=r.Diphtheria, measles=r.Measles, bmi=r.BMI,
    incidents_hiv=r.Incidents_HIV, alcohol_consumption=r.Alcohol_consumption,
    thinness_10_19=r.Thinness_ten_nineteen_years,
    thinness_5_9=r.Thinness_five_nine_years,
    gdp_per_capita=r.GDP_per_capita, population_mln=r.Population_mln,
    schooling=r.Schooling, economy_status_developed=r.Economy_status_Developed), 2))

print('minimal :', round(predict_life_expectancy_min(
    region=r.Region, year=r.Year, gdp_per_capita=r.GDP_per_capita,
    population_mln=r.Population_mln, schooling=r.Schooling,
    economy_status_developed=r.Economy_status_Developed), 2))